In [1]:
"""
        DESTRUCTOR IN PYTHON
=========================================

This file explains destructors in Python in detail.
It covers:
1. What a destructor is
2. Difference between constructor and destructor
3. Automatic execution of __del__
4. Reference counting
5. Garbage collection
6. Real use cases
7. Employee example
8. Why destructors are risky for critical cleanup
9. Circular references and destructors
10. Exceptions inside destructors
11. Why time.sleep() and gc.collect() are useful for testing
12. Interview / exam-style questions and answers

Run this file directly, or split it into notebook cells.
Every section is written with heavy comments so the theory and behavior are clear.
"""

import gc
import time



In [2]:

# ============================================================
# 1) WHAT IS A DESTRUCTOR?
# ============================================================
#
# A destructor is a special method that runs when an object is about to be
# destroyed / cleaned up.
#
# In Python, the destructor method is:
#     __del__(self)
#
# The constructor __init__ initializes an object.
# The destructor __del__ is used for cleanup before the object disappears.
#
# Important:
# - You usually do NOT call __del__ manually.
# - Python tries to call it automatically when the object is no longer needed.
# - In practice, __del__ is not the most reliable way to manage important
#   resources, but it is still important to understand.


def show_destructor_intro():
    print("=" * 80)
    print("1. WHAT IS A DESTRUCTOR?")
    print("=" * 80)
    print("Constructor (__init__) -> initialize object")
    print("Destructor  (__del__)  -> cleanup before object removal")
    print("Python may call __del__ automatically when object is being destroyed")
    print()


show_destructor_intro()



1. WHAT IS A DESTRUCTOR?
Constructor (__init__) -> initialize object
Destructor  (__del__)  -> cleanup before object removal
Python may call __del__ automatically when object is being destroyed



# 🔥 Core Question

👉 **When reference count becomes 0:**
* Does object get destroyed first?
* Or `__del__()` runs first?
* Who actually destroys the object?

## ✅ FINAL ANSWER (Short)

* 👉 **Destructor (`__del__`) runs before the object is fully destroyed**
* 👉 **Garbage Collector / Python runtime** is responsible for destroying the object
* 👉 **Destructor does NOT destroy the object** — it only performs cleanup

---

## 🧠 INTERNAL STEP-BY-STEP FLOW

When reference count becomes 0:

1. Python detects `ref_count == 0`
2. Python calls object's destructor (`__del__`)
3. Destructor code executes
4. After `__del__` finishes → memory is freed
5. Object is finally destroyed

### 🎯 So Order is:

```text
ref_count = 0
   ↓
__del__() runs FIRST
   ↓
object memory is deallocated (destroyed)
```

### 🚀 SCENARIO 1: Simple Example

```python
class Demo:
    def __init__(self):
        print("Object created")

    def __del__(self):
        print("Destructor running")

obj = Demo()
del obj
print("End of program")
```

**🔍 Step-by-step Execution:**

* **Step 1:** `obj = Demo()` → `ref_count = 1`
* **Step 2:** `del obj` → `ref_count = 0`
* **Step 3:** Python immediately calls: `__del__()` → prints: "Destructor running"
* **Step 4:** Memory is freed → object destroyed
* **Step 5:** `print("End of program")`

**✅ Output:**

`Object created`
`Destructor running`
`End of program`

### 🚀 SCENARIO 2: Proving Destructor Runs Before Destruction

```python
class Demo:
    def __del__(self):
        print("Destructor running")
        print(self)   # object still exists here!

obj = Demo()
del obj
```

**🔍 Key Observation:**
* 👉 Inside `__del__`, object is still accessible!
* 💡 **Meaning:** Object is **NOT** destroyed yet. Destructor runs **before** destruction.

### 🚀 SCENARIO 3: Multiple References

```python
class Demo:
    def __del__(self):
        print("Destroyed")

obj1 = Demo()
obj2 = obj1

del obj1   # ref_count = 1
del obj2   # ref_count = 0 → trigger
```

**🔍 Flow:**
* `obj1` deleted → nothing happens
* `obj2` deleted → `ref_count = 0`
* → `__del__` runs
* → object destroyed

### 🚀 SCENARIO 4: Function Scope

```python
class Demo:
    def __del__(self):
        print("Destructor called")

def test():
    obj = Demo()
    print("Inside function")

test()
print("Outside function")
```

**🔍 Flow:**
* Function ends → `obj` goes out of scope
* → `ref_count = 0`
* → `__del__` runs
* → object destroyed

### 🚀 SCENARIO 5: Garbage Collector Case (Circular Reference)

```python
import gc

class A:
    def __del__(self):
        print("A destroyed")

class B:
    def __del__(self):
        print("B destroyed")

a = A()
b = B()

a.ref = b
b.ref = a

del a
del b

gc.collect()
```

**🔍 Important Difference Here:**
* 💡 **What happens:** Reference count never becomes 0 ❌
* **Garbage Collector** detects cycle ✅

**Flow:**
1. GC finds unreachable objects
2. → Calls destructors (`__del__`)
3. → Frees memory

### ⚠️ VERY IMPORTANT CONCEPT

**❗ Who destroys object?**

| Component | Role |
|-----------|------|
| `__del__()` | Cleanup logic (user-defined) |
| Python runtime / GC | Actual memory deallocation |

**🧠 Think Like This:**
* 👉 Destructor = **"Last wish before dying"**
* 👉 Garbage Collector = **"Actual executioner"**

### 🔥 FINAL VISUAL FLOW

```plaintext
Reference count = 0
        ↓
Python runtime triggers __del__()
        ↓
Your cleanup code runs
        ↓
Python frees memory
        ↓
Object completely destroyed
```

### 🎯 ONE-LINE INTERVIEW ANSWER

👉 **"When reference count becomes zero, Python first calls `__del__()` for cleanup, and after it finishes, the garbage collector/runtime deallocates the object memory."**

In [3]:

# ============================================================
# 2) CONSTRUCTOR VS DESTRUCTOR
# ============================================================
#
# Constructor __init__:
# - runs when object is created
# - sets up attributes
# - prepares object for use
#
# Destructor __del__:
# - runs when object is being removed from memory (before actual removal just before)
# - used for cleanup
# - may close files, release handles, or print final messages
#
# In most programs, constructor is almost always used.
# Destructor is used only when you need cleanup logic.


class DemoLifecycle:
    def __init__(self, name):
        self.name = name
        print(f"__init__ called: object {self.name} created")

    def __del__(self):
        # This may run when Python destroys the object.
        print(f"__del__ called: object {self.name} is being destroyed")


def show_constructor_vs_destructor():
    print("=" * 80)
    print("2. CONSTRUCTOR VS DESTRUCTOR")
    print("=" * 80)

    obj = DemoLifecycle("A")
    print("Object is in use")
    print("Leaving the function / deleting reference may trigger __del__")
    print()


show_constructor_vs_destructor()



2. CONSTRUCTOR VS DESTRUCTOR
__init__ called: object A created
Object is in use
Leaving the function / deleting reference may trigger __del__

__del__ called: object A is being destroyed


In [4]:

# ============================================================
# 3) WHEN DOES __del__ RUN AUTOMATICALLY?
# ============================================================
#
# Python uses reference counting in CPython.
# Every object keeps track of how many references point to it.
#
# Example:
#   a = MyClass()
#   b = a
#
# Now the same object has two references: a and b.
# If you delete one reference with del a, the object is not destroyed yet,
# because b still points to it.
#
# When the final reference is removed, the object becomes unreachable.
# Then Python may destroy it and call __del__.
#
# But note:
# - Exact timing can depend on the Python implementation.
# - CPython often destroys quickly because of reference counting.
# - Other Python implementations may behave differently.


class ReferenceDemo:
    def __init__(self, label):
        self.label = label
        print(f"ReferenceDemo {self.label} created")

    def __del__(self):
        print(f"ReferenceDemo {self.label} destroyed")


def show_automatic_execution():
    print("=" * 80)
    print("3. WHEN DOES __del__ RUN AUTOMATICALLY?")
    print("=" * 80)

    a = ReferenceDemo("X")
    b = a
    print("Two references exist: a and b")
    print("Deleting a only...")
    del a
    print("Destructor should NOT run yet because b still exists")
    print("Deleting b now...")
    del b
    print("Now the object has no references, so __del__ may run")
    print()


show_automatic_execution()



3. WHEN DOES __del__ RUN AUTOMATICALLY?
ReferenceDemo X created
Two references exist: a and b
Deleting a only...
Destructor should NOT run yet because b still exists
Deleting b now...
ReferenceDemo X destroyed
Now the object has no references, so __del__ may run



In [5]:

# ============================================================
# 4) REFERENCE COUNTING EXPLAINED
# ============================================================
#
# Reference counting means Python tracks how many names refer to an object.
#
# Simplified idea:
# - Every new assignment adds a reference
# - del removes a reference
# - When reference count becomes zero, object becomes eligible for cleanup
#
# You can inspect reference counts using sys.getrefcount(obj), but note that
# getrefcount itself temporarily adds a reference.
# For teaching purposes, we will use a simple conceptual example instead of
# depending on exact numeric counts.


def show_reference_counting_explanation():
    print("=" * 80)
    print("4. REFERENCE COUNTING")
    print("=" * 80)
    print("Python keeps track of how many references point to an object.")
    print("When the last reference disappears, the object can be destroyed.")
    print("That is one reason __del__ may run automatically.")
    print()


show_reference_counting_explanation()



4. REFERENCE COUNTING
Python keeps track of how many references point to an object.
When the last reference disappears, the object can be destroyed.
That is one reason __del__ may run automatically.



In [6]:

# ============================================================
# 5) WHY USE DESTRUCTORS?
# ============================================================
#
# Destructors are useful for cleanup tasks such as:
# - closing file handles
# - releasing locks
# - cleaning temporary resources
# - closing network connections
# - final logging
#
# In real-world programming, these tasks are often better handled by:
# - context managers (with statement)
# - explicit close() methods
# - try/finally blocks
#
# Still, __del__ is useful to understand because it shows object lifecycle.


class FileLikeResource:
    def __init__(self, resource_name):
        self.resource_name = resource_name
        self.is_open = True
        print(f"Resource '{self.resource_name}' opened")

    def close(self):
        if self.is_open:
            print(f"Resource '{self.resource_name}' closed manually")
            self.is_open = False

    def __del__(self):
        # Cleanup should be safe and small.
        # Avoid heavy logic inside __del__.
        if getattr(self, "is_open", False):
            print(f"__del__: auto-closing resource '{self.resource_name}'")
            self.is_open = False


def show_use_cases():
    print("=" * 80)
    print("5. WHY USE DESTRUCTORS?")
    print("=" * 80)

    res = FileLikeResource("temp_connection")
    res.close()
    print("Deleting object reference now...")
    del res
    print("If resource was already closed, destructor has little to do")
    print()


show_use_cases()



5. WHY USE DESTRUCTORS?
Resource 'temp_connection' opened
Resource 'temp_connection' closed manually
Deleting object reference now...
If resource was already closed, destructor has little to do



In [7]:

# ============================================================
# 6) EMPLOYEE EXAMPLE
# ============================================================
#
# This is a classic destructor example.
#
# The object is created, used, and then deleted.
# When the last reference disappears, __del__ can run.
#
# This example is mainly educational, so we print messages to observe the lifecycle.


class Employee:
    def __init__(self, eid, name, salary):
        self.emp_id = eid
        self.emp_name = name
        self.emp_salary = salary
        print(f"Employee created -> ID: {self.emp_id}, Name: {self.emp_name}")

    def display(self):
        print("Employee id    :", self.emp_id)
        print("Employee name  :", self.emp_name)
        print("Employee salary:", self.emp_salary)

    def __del__(self):
        print(f"__del__ called for Employee {self.emp_name} (ID: {self.emp_id})")


def show_employee_example():
    print("=" * 80)
    print("6. EMPLOYEE EXAMPLE")
    print("=" * 80)

    e1 = Employee(101, "Shantanu", 50000)
    e1.display()
    print("Deleting employee object...")
    del e1
    print("After del e1, destructor may run immediately in CPython")
    print()


show_employee_example()



6. EMPLOYEE EXAMPLE
Employee created -> ID: 101, Name: Shantanu
Employee id    : 101
Employee name  : Shantanu
Employee salary: 50000
Deleting employee object...
__del__ called for Employee Shantanu (ID: 101)
After del e1, destructor may run immediately in CPython



In [16]:

# ============================================================
# 7) USING TIME MODULE FOR TESTING DESTRUCTOR BEHAVIOR
# ============================================================
#
# time.sleep() is useful when you want to observe output timing clearly.
#
# It is not required for destructors themselves.
# It is just a teaching/testing tool so you can see which print happens first.
#
# Example flow:
# - create object
# - wait a little
# - delete object
# - wait again
#
# This makes the lifecycle easier to observe.


class TimedObject:
    def __init__(self, name):
        self.name = name
        print(f"TimedObject {self.name} created")

    def __del__(self):
        print(f"TimedObject {self.name} destroyed")


def show_time_module_test():
    print("=" * 80)
    print("7. USING TIME MODULE FOR TESTING")
    print("=" * 80)

    obj = TimedObject("T1")
    print("Sleeping for 1 second...")
    time.sleep(1)
    print("Deleting object now...")
    del obj
    print("Sleeping again for 1 second...")
    time.sleep(1)
    print("Done")
    print()


show_time_module_test()



7. USING TIME MODULE FOR TESTING
TimedObject T1 created
Sleeping for 1 second...
Deleting object now...
TimedObject T1 destroyed
Sleeping again for 1 second...
Done



In [9]:

# ============================================================
# 8) FORCING GARBAGE COLLECTION WITH gc.collect()
# ============================================================
#
# Python has a garbage collector that handles objects, especially those involved
# in cycles.
#
# In many cases, reference counting is enough.
# But in cyclic references, objects may not be cleaned immediately.
#
# gc.collect() can force a garbage collection pass.
# This is useful in demonstrations, testing, and debugging.


def show_gc_collect_demo():
    print("=" * 80)
    print("8. FORCING GARBAGE COLLECTION")
    print("=" * 80)

    class Temp:
        def __del__(self):
            print("Temp object destroyed")

    t = Temp()
    del t
    print("Calling gc.collect() to force garbage collector work")
    gc.collect()
    print()


show_gc_collect_demo()



8. FORCING GARBAGE COLLECTION
Temp object destroyed
Calling gc.collect() to force garbage collector work



In [10]:

# ============================================================
# 9) LIMITATION: DESTRUCTORS MAY BE UNPREDICTABLE
# ============================================================
#
# One disadvantage of __del__ is that the exact time of execution may be hard
# to predict in all cases.
#
# Reasons:
# - multiple references may still exist
# - cyclic references may delay cleanup
# - interpreter shutdown can affect cleanup order
# - different Python implementations may behave differently
#
# This is why relying on __del__ for mission-critical cleanup is risky.


class UnpredictableDemo:
    def __init__(self, name):
        self.name = name
        print(f"UnpredictableDemo {self.name} created")

    def __del__(self):
        print(f"UnpredictableDemo {self.name} destroyed")


def show_unpredictable_behavior():
    print("=" * 80)
    print("9. LIMITATION: UNPREDICTABLE EXECUTION")
    print("=" * 80)

    x = UnpredictableDemo("A")
    y = x
    z = y
    print("Multiple references exist: x, y, z")
    print("Deleting x")
    del x
    print("Destructor should not run yet because y and z still exist")
    print("Deleting y")
    del y
    print("Destructor should still not run because z exists")
    print("Deleting z")
    del z
    print("Now the object can be destroyed")
    print()


show_unpredictable_behavior()



9. LIMITATION: UNPREDICTABLE EXECUTION
UnpredictableDemo A created
Multiple references exist: x, y, z
Deleting x
Destructor should not run yet because y and z still exist
Deleting y
Destructor should still not run because z exists
Deleting z
UnpredictableDemo A destroyed
Now the object can be destroyed



In [11]:

# ============================================================
# 10) CIRCULAR REFERENCES AND DESTRUCTORS
# ============================================================
#
# A circular reference happens when object A refers to object B and object B
# refers back to object A.
#
# Example:
#   obj1.other = obj2
#   obj2.other = obj1
#
# In older Python behavior, cycles could make cleanup tricky.
# Modern Python garbage collector can detect many cycles.
#
# However, __del__ complicates things because finalization order can be tricky.
# This is one reason destructors are not always ideal for complex resource cleanup.


class Node:
    def __init__(self, name):
        self.name = name
        self.other = None
        print(f"Node {self.name} created")

    def __del__(self):
        print(f"Node {self.name} destroyed")


def show_circular_reference_issue():
    print("=" * 80)
    print("10. CIRCULAR REFERENCES")
    print("=" * 80)

    a = Node("A")
    b = Node("B")
    a.other = b
    b.other = a

    print("Circular reference created: A <-> B")
    print("Deleting a and b references...")
    del a
    del b
    print("The garbage collector may handle this later")
    print("Calling gc.collect() to encourage cleanup")
    gc.collect()
    print()


show_circular_reference_issue()



10. CIRCULAR REFERENCES
Node A created
Node B created
Circular reference created: A <-> B
Deleting a and b references...
The garbage collector may handle this later
Calling gc.collect() to encourage cleanup
Node A destroyed
Node B destroyed



In [12]:

# ============================================================
# 11) CAN EXCEPTIONS OCCUR INSIDE A DESTRUCTOR?
# ============================================================
#
# Yes, exceptions can happen inside __del__.
# But they are dangerous because they are hard to manage during cleanup.
#
# Best practice:
# - keep __del__ simple
# - use try/except inside __del__ if needed
# - avoid risky operations in destructors
#
# A destructor should never depend on fragile state.


class SafeDestructor:
    def __init__(self, value):
        self.value = value
        print(f"SafeDestructor created with value={self.value}")

    def __del__(self):
        try:
            # Example of safe cleanup logic.
            # Even if something is missing, we catch it.
            print(f"SafeDestructor cleaning value={self.value}")
        except Exception as exc:
            print("Exception inside __del__ was handled safely:", exc)


def show_exception_in_destructor():
    print("=" * 80)
    print("11. EXCEPTIONS INSIDE DESTRUCTORS")
    print("=" * 80)

    obj = SafeDestructor(100)
    del obj
    print("Destructor executed with protected cleanup")
    print()


show_exception_in_destructor()



11. EXCEPTIONS INSIDE DESTRUCTORS
SafeDestructor created with value=100
SafeDestructor cleaning value=100
Destructor executed with protected cleanup



In [13]:

# ============================================================
# 12) WHY DESTRUCTORS ARE NOT ALWAYS THE BEST CHOICE
# ============================================================
#
# Python developers usually prefer:
# - context managers (with open(...))
# - explicit close() methods
# - try/finally
#
# Why?
# Because __del__ timing is not fully under your control.
#
# For example, file cleanup should usually be done like:
#
#   with open("file.txt") as f:
#       data = f.read()
#
# The file closes automatically when the block ends.
# This is more reliable than depending on __del__.


def show_best_practice_note():
    print("=" * 80)
    print("12. BEST PRACTICE NOTE")
    print("=" * 80)
    print("For important cleanup, prefer:")
    print("- with statement")
    print("- close() methods")
    print("- try/finally")
    print("Use __del__ mainly for learning and lightweight cleanup support")
    print()


show_best_practice_note()



12. BEST PRACTICE NOTE
For important cleanup, prefer:
- with statement
- close() methods
- try/finally
Use __del__ mainly for learning and lightweight cleanup support



In [14]:

# ============================================================
# 13) ANSWERS TO YOUR QUESTIONS
# ============================================================
#
# Q1) When does a destructor run automatically?
# A1) When the object has no more references and is ready to be cleaned up.
#     In CPython, that often happens immediately due to reference counting.
#
# Q2) What is reference counting in Python?
# A2) Python keeps track of how many names/references point to an object.
#     When the count becomes zero, the object can be destroyed.
#
# Q3) Why use destructors for database connections?
# A3) To release resources if the object is discarded unexpectedly.
#     But in real applications, explicit close methods or context managers are
#     usually safer and more predictable.
#
# Q4) Why does circular referencing break destructors?
# A4) Cycles can delay or complicate finalization, because objects refer to each
#     other even after external references are deleted.
#
# Q5) Can exceptions occur inside a destructor?
# A5) Yes. That is why __del__ should be simple and protected with try/except.
#
# Q6) Why use the time module in testing?
# A6) To pause execution and clearly observe when messages print during object
#     creation, deletion, and cleanup.


def show_final_answers():
    print("=" * 80)
    print("13. FINAL ANSWERS")
    print("=" * 80)
    print("1) Destructor runs when object is being cleaned up.")
    print("2) Reference counting tracks how many references point to an object.")
    print("3) Database/file cleanup is a possible use, but not the safest primary method.")
    print("4) Circular references complicate cleanup.")
    print("5) Exceptions can happen inside __del__, so keep it safe.")
    print("6) time.sleep() helps observe destructor timing in demos.")
    print()


show_final_answers()



13. FINAL ANSWERS
1) Destructor runs when object is being cleaned up.
2) Reference counting tracks how many references point to an object.
3) Database/file cleanup is a possible use, but not the safest primary method.
4) Circular references complicate cleanup.
5) Exceptions can happen inside __del__, so keep it safe.
6) time.sleep() helps observe destructor timing in demos.



In [15]:

# ============================================================
# 14) QUICK RECAP
# ============================================================
#
# Destructor in Python:
# - special method __del__
# - runs when object is being destroyed
# - useful for cleanup
# - not always predictable
# - avoid relying on it for critical resource management
#
# The safest Pythonic cleanup tools are usually context managers and explicit
# close methods.


print("=" * 80)
print("QUICK RECAP")
print("Destructor = __del__")
print("Used for cleanup when object is destroyed")
print("Triggered automatically, not usually manually")
print("Reference counting and garbage collection affect timing")
print("Prefer with/close() for important resources")
print("=" * 80)


QUICK RECAP
Destructor = __del__
Used for cleanup when object is destroyed
Triggered automatically, not usually manually
Reference counting and garbage collection affect timing
Prefer with/close() for important resources
